# Plots for Descriptive Stats Features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
notebook_dir = Path().resolve()
sys.path.append(str(notebook_dir.parent))        # ../
sys.path.append(str(notebook_dir.parent.parent)) # ../../


import pandas as pd
import numpy as np
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go



from plot_helper.colors import COLORS


## Sleep
### Get Data

In [ ]:
df_sleep = pd.read_csv("../../output/1_feature_extraction/df_features_sleep_2026-07-08.csv")
display(df_sleep.head())
df_sleep_stages = pd.read_csv("../../output/1_feature_extraction/df_features_sleep_stages_2026-07-08.csv")
display(df_sleep_stages.head())


n = df_sleep['study_id'].nunique()

# Convert time columns to seconds
time_columns = ['median_sleep_onset', 'median_midpoint', 'median_wakeup']
for col in time_columns:
    df_sleep[col + '_seconds'] = pd.to_datetime(df_sleep[col], format='mixed').dt.time.apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )
display(df_sleep[time_columns + [col + '_seconds' for col in time_columns]].head())

# Midnight crossover adjustment
df_sleep['median_sleep_onset_seconds_adjusted'] = df_sleep['median_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df_sleep['median_midpoint_seconds_adjusted'] = df_sleep['median_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)


# Keep as datetime for plotting (plotly handles datetime axes natively)
df_sleep['median_onset_plot'] = df_sleep['median_sleep_onset_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

df_sleep['median_midpoint_plot'] = df_sleep['median_midpoint_seconds_adjusted'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)
df_sleep['median_wakeup_plot'] = df_sleep['median_wakeup_seconds'].apply(
    lambda x: pd.Timestamp('2000-01-01') + pd.Timedelta(seconds=x)
)

def seconds_to_clock_hhmm(seconds):
    if pd.isna(seconds):
        return ""
    seconds = int(seconds) % (24 * 3600)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"


df_sleep["median_sleep_onset_hhmm"] = (
    df_sleep["median_sleep_onset_seconds_adjusted"]
    .apply(seconds_to_clock_hhmm)
)
df_sleep["median_sleep_midpoint_hhmm"] = (
    df_sleep["median_midpoint_seconds_adjusted"]
    .apply(seconds_to_clock_hhmm)
)
df_sleep["median_wakeup_hhmm"] = (
    df_sleep["median_wakeup_seconds"]
    .apply(seconds_to_clock_hhmm)
)



### Plot

In [ ]:
""" plot each feature as one instead of single patients; 2 week summary for each feature """

fig = make_subplots(
    rows=2, cols=6,
    specs=[
        [{},{}, {}, {}, {}, {}],  
        [{},{}, {}, {}, {}, None],  

    ],
    vertical_spacing=0.1,
    subplot_titles=(
        f"Sleep Duration [h]",
        f"WASO [min]",
        f"WASO Count",
        f"Sleep Onset [hh:mm]",
        f"Sleep Midpoint [hh:mm]",
        f"Wakeup [hh:mm]",
        f"Light Sleep Duration [h]",
        f"Deep Sleep Duration [h]",
        f"REM Sleep Duration [h]",
        f"Overall Sleep Score",
        f"SER [%]",
    ),
)

fig.update_annotations(font_size=14)

fig.add_trace(
    go.Box(
        y = df_sleep['median_sleep_duration'],
        name='Daily Sleep Duration',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep['median_sleep_duration'])],
        x=['Daily Sleep Duration'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        
    ),
    row=1, col=1
)
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="right",
    x=1,

))
#*WASO in minutes
fig.add_trace(
    go.Box(
        y = df_sleep['median_waso'],
        name='WASO',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep['median_waso'])],
        x=['WASO'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=2
)

#*WASO counts
fig.add_trace(
    go.Box(
        y = df_sleep['median_awake'],
        name='Awake Count',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep['median_awake'])],
        x=['Awake Count'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=3
)

#*SER in minutes
fig.add_trace(
    go.Box(
        y = df_sleep['median_ser'],
        name='SER',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=5
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep['median_ser'])],
        x=['SER'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=5
)

sleep_time_tickvals = [
    20 * 3600,  # 20:00
    21 * 3600,  # 21:00
    22 * 3600,  # 22:00
    23 * 3600,  # 23:00
    24 * 3600,  # 00:00
    25 * 3600,  # 01:00
    26 * 3600,  # 02:00
    27 * 3600,  # 03:00
    28 * 3600,  # 04:00
    29 * 3600,  # 05:00
    30*3600,  # 06:00
]
#*Sleep Onset
fig.add_trace(
    go.Box(
        y = df_sleep['median_sleep_onset_seconds_adjusted'],
        name='Sleep Onset',
        boxpoints='all',
        customdata=df_sleep["median_sleep_onset_hhmm"],
        hovertemplate="Sleep Onset: %{customdata}<extra></extra>",
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=4
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_sleep['median_sleep_onset_seconds_adjusted']))],
        x=['Sleep Onset'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=4
)

fig.update_yaxes(
    tickmode="array",
    tickvals=sleep_time_tickvals,
    ticktext=[seconds_to_clock_hhmm(x) for x in sleep_time_tickvals],
    row=1,
    col=4
)
fig.update_yaxes(tickformat="%H:%M", row=1, col=4)

#*Midpoint of Sleep
fig.add_trace(
    go.Box(
        y = df_sleep['median_midpoint_seconds_adjusted'],
        name='Sleep Midpoint',
        boxpoints='all',
        customdata=df_sleep["median_sleep_midpoint_hhmm"],
        hovertemplate="Sleep Midpoint: %{customdata}<extra></extra>",
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=5
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_sleep['median_midpoint_seconds_adjusted']))],
        x=['Sleep Midpoint'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=5
)
fig.update_yaxes(
    tickmode="array",
    tickvals=sleep_time_tickvals,
    ticktext=[seconds_to_clock_hhmm(x) for x in sleep_time_tickvals],
    row=1,
    col=5
)

fig.update_yaxes(tickformat="%H:%M", row=1, col=5)

#* Wakeup
wakeup_tickvals = [
    4 * 3600,   # 04:00
    5 * 3600,   # 05:00
    6 * 3600,   # 06:00
    7 * 3600,   # 07:00
    8 * 3600,   # 08:00
    9 * 3600,   # 09:00
    10 * 3600,  # 10:00
    11 * 3600,  # 11:00
]

fig.add_trace(
    go.Box(
        y = df_sleep['median_wakeup_seconds'],
        name='Wakeup',
        boxpoints='all',
        customdata=df_sleep["median_wakeup_hhmm"],
        hovertemplate="Wakeup: %{customdata}<extra></extra>",
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=6
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep['median_wakeup_seconds'])],
        x=['Wakeup'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=6
)
fig.update_yaxes(
    tickmode="array",
    tickvals=wakeup_tickvals,
    ticktext=[seconds_to_clock_hhmm(x) for x in wakeup_tickvals],
    row=1,
    col=6
)
fig.update_yaxes(tickformat="%H:%M", row=1, col=6)

#* Overall Sleep Score
fig.add_trace(
    go.Box(
        y = df_sleep_stages['median_overall_sleep_score'],
        name='Overall Sleep Score',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=4
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep_stages['median_overall_sleep_score'])],
        x=['Overall Sleep Score'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=4
)


#* Light Sleep Duration
fig.add_trace(
    go.Box(
        y = df_sleep_stages['median_light_sleep_duration'],
        name='Light Sleep Duration',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep_stages['median_light_sleep_duration'])],
        x=['Light Sleep Duration'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=1
)
#* Deep Sleep Duration
fig.add_trace(
    go.Box(
        y = df_sleep_stages['median_deep_sleep_duration'],
        name='Deep Sleep Duration',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=2
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep_stages['median_deep_sleep_duration'])],
        x=['Deep Sleep Duration'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=2
)

#* REM Sleep Duration
fig.add_trace(
    go.Box(
        y = df_sleep_stages['median_rem_sleep_duration'],
        name='REM Sleep Duration',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=3
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_sleep_stages['median_rem_sleep_duration'])],
        x=['REM Sleep Duration'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=3
)
# --- Layout ---
fig.update_layout(
    title_text=f"Sleep Features, n = {n}",
    height=800,
    width=1800,
    font=dict(size=12),
    plot_bgcolor="rgba(0,0,0,0)",
)

fig.show()

#save plot as png
date = datetime.now().strftime("%Y-%m-%d")
fig.write_image(f"../../plots/Features/sleep_features_box_plot_{date}.png", width=1800, height=800, scale=3)


## Steps
### Get Data

In [ ]:
df_step = pd.read_csv("../../output/1_feature_extraction/df_features_step_2026-07-08.csv")
display(df_step.head())

n_step = df_step['study_id'].nunique()
display(n_step)
display(df_step.columns)

#!exclude DEC_46 from all step features
df_step = df_step[df_step['study_id'] != 'DEC_46']



n_step = df_step['study_id'].nunique()
display(n_step)

display(df_step.sort_values(by='median_sedentary_time_h', ascending=False).head(10))

### Plot

In [ ]:
""" plot each feature as one instead of single patients; 2 week summary for each feature """

fig = make_subplots(
    rows=2, cols=4,
    specs=[
        [{},{}, {}, {}],  
        [{},{}, {}, {}],  

    ],
    vertical_spacing=0.1,
    subplot_titles=(
        f"Steps [count]",
        f"Sedentary Time [h]",
        f"Movement Time [h]",
        f"M/S ratio",
        f"Steps 2h after Wakeup",
        f"Steps 4h after Wakeup",
        f"Steps 2h before Onset",
        f"Steps 4h before Onset",

    ),
)

fig.update_annotations(font_size=14)

#* Steps
fig.add_trace(
    go.Box(
        y = df_step['median_steps'],
        name='Steps',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_step['median_steps'])],
        x=['Steps'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        
    ),
    row=1, col=1
)
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="right",
    x=1,

))
#*Sedentary time
fig.add_trace(
    go.Box(
        y = round(df_step['median_sedentary_time_h'], 2),
        name='Sedentary Time',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(round(df_step['median_sedentary_time_h'], 2))],
        x=['Sedentary Time'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=2
)

#*Movement Time
fig.add_trace(
    go.Box(
        y = round(df_step['median_movement_time_h'], 2),
        name='Movement Time',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(round(df_step['median_movement_time_h'], 2))],
        x=['Movement Time'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=3
)

#*M/S Ratio
fig.add_trace(
    go.Box(
        y = df_step['median_ratio'],
        name='M/S Ratio',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=4
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_step['median_ratio'])],
        x=['M/S Ratio'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=4
)

#*Steps 2h after Wakeup
fig.add_trace(
    go.Box(
        y = df_step['median_steps_2h'],
        name='Steps 2h after Wakeup',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_step['median_steps_2h']))],
        x=['Steps 2h after Wakeup'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=1
)


#*Steps 4h after Wakeup
fig.add_trace(
    go.Box(
        y = df_step['median_steps_4h'],
        name='Steps 4h after Wakeup',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=2
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_step['median_steps_4h']))],
        x=['Steps 4h after Wakeup'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=2
)

#*Steps 2h before Onset
fig.add_trace(
    go.Box(
        y = df_step['median_steps_onset_2h'],
        name='Steps 2h before Onset',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=3
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_step['median_steps_onset_2h']))],
        x=['Steps 2h before Onset'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=3
)

#*Steps 4h before Onset
fig.add_trace(
    go.Box(
        y = df_step['median_steps_onset_4h'],
        name='Steps 4h before Onset',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=2, col=4
)

fig.add_trace(
    go.Scatter(
        y=[(np.mean(df_step['median_steps_onset_4h']))],
        x=['Steps 4h before Onset'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=2, col=4
)



# --- Layout ---
fig.update_layout(
    title_text=f"Activity Features, n = {n_step}",
    height=800,
    width=1800,
    font=dict(size=12),
    plot_bgcolor="rgba(0,0,0,0)",
)

fig.show()

#save plot as png
date = datetime.now().strftime("%Y-%m-%d")
fig.write_image(f"../../plots/Features/step_features_box_plot_{date}.png", width=1800, height=800, scale=3)


## HR/HRV
### Get Data

In [ ]:
df_hr = pd.read_csv("../../output/1_feature_extraction/df_features_hr_2026-07-08.csv")
display(df_hr.head())
n_hr = df_hr['study_id'].nunique()
print(f"Number of unique study IDs in df_hr: {n}")

df_nocturnal_hr = pd.read_csv("../../output/1_feature_extraction/df_features_nocturnal_hr_2026-07-08.csv")
display(df_nocturnal_hr.head())
n_nocturnal = df_nocturnal_hr['study_id'].nunique()
print(f"Number of unique study IDs in df_nocturnal_hr: {n_nocturnal}")

#check which patients have less than 7 days/nights of data
df_hr_exclude = df_hr[df_hr['n_days'] < 7]
print("Patients with less than 7 days of data:")
display(df_hr_exclude)
print(len(df_hr_exclude['study_id'].unique()))
df_nocturnal_hr_exclude = df_nocturnal_hr[df_nocturnal_hr['n_nights'] < 7]
print("Patients with less than 7 nights of data:")
display(df_nocturnal_hr_exclude)
print(len(df_nocturnal_hr_exclude['study_id'].unique()))

#exclude patients with less than 7 days/nights of data
df_hr = df_hr[df_hr['n_days'] >= 7]
df_nocturnal_hr = df_nocturnal_hr[df_nocturnal_hr['n_nights'] >= 7]
n = df_hr['study_id'].nunique()
n_nocturnal = df_nocturnal_hr['study_id'].nunique()


display(df_hr.shape)
display(df_nocturnal_hr.shape)

display(df_hr.columns)
display(df_nocturnal_hr.columns)


### Plots

In [ ]:
""" plot each feature as one instead of single patients; 2 week summary for each feature """

fig = make_subplots(
    rows=1, cols=4,
    specs=[
        [{},{}, {}, {}],  
       
    ],
    vertical_spacing=0.1,
    subplot_titles=(
        f"HR [bpm], n = {n_hr}",
        f"HRV (RMSSD) [ms], n = {n_hr}",
        f"Nocturnal HR [bpm], n = {n_nocturnal}",
        f"Nocturnal HRV (RMSSD) [ms], n = {n_nocturnal}",


    ),
)

fig.update_annotations(font_size=14)

#* HR
fig.add_trace(
    go.Box(
        y = df_hr['mean_hr'],
        name='HR',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_hr['mean_hr'])],
        x=['HR'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        
    ),
    row=1, col=1
)
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.1,
    xanchor="right",
    x=1,

))
#*HRV (RMSSD)
fig.add_trace(
    go.Box(
        y = df_hr['mean_rmssd'],
        name='HRV (RMSSD)',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_hr['mean_rmssd'])],
        x=['HRV (RMSSD)'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=2
)

#*Nocturnal HR
fig.add_trace(
    go.Box(
        y = df_nocturnal_hr['mean_hr'],
        name='Nocturnal HR',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_nocturnal_hr['mean_hr'])],
        x=['Nocturnal HR'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=3
)

#*Nocturnal HRV (RMSSD)
fig.add_trace(
    go.Box(
        y = df_nocturnal_hr['mean_rmssd'],
        name='Nocturnal HRV (RMSSD)',
        boxpoints='all',
        marker_color=COLORS.get("blue1"),
        showlegend=False
    ),
    row=1, col=4
)

fig.add_trace(
    go.Scatter(
        y=[np.mean(df_nocturnal_hr['mean_rmssd'])],
        x=['Nocturnal HRV (RMSSD)'],
        mode='markers',
        name=f'Mean',
        marker=dict(color=COLORS.get("red1")),
        showlegend=False
        
    ),
    row=1, col=4
)



# --- Layout ---
fig.update_layout(
    title_text=f"HR and HRV Features",
    height=500,
    width=1800,
    font=dict(size=12),
    plot_bgcolor="rgba(0,0,0,0)",
)

fig.show()

#save plot as png
date = datetime.now().strftime("%Y-%m-%d")
fig.write_image(f"../../plots/Features/hr_hrv_features_box_plot_{date}.png", width=1800, height=500, scale=3)
